# Bronze: Ingestão Incremental de Dados do Bitcoin

Este notebook realiza a ingestão dos dados brutos de mercado do Bitcoin (BTC/USD) na camada Bronze da arquitetura Medallion.

Os dados são obtidos diariamente via Azure Data Factory, que utiliza a API pública da CoinGecko para extrair preços, capitalização de mercado e volume de negociação, salvando o resultado no container landing do ADLS Gen2.

Na Bronze, os dados são carregados exatamente como vieram da fonte, sem transformações, e persistidos no formato Delta Lake. A cada execução, o notebook 
verifica a data do último registro já armazenado e realiza uma carga incremental via MERGE, inserindo apenas os dados novos e evitando duplicações.

Colunas adicionadas nesta camada:
- ativo: identificador do ativo (BTC)
- moeda: moeda de referência (USD)
- data_carga_lake: data em que o dado foi carregado no Data Lake
- fonte: origem dos dados (coingecko)



In [0]:
%pip install azure-storage-blob

In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("env", "prod")
dbutils.widgets.text("execution_date", "")

env            = dbutils.widgets.get("env")
execution_date = dbutils.widgets.get("execution_date")

print(f"Ambiente: {env}")
print(f"Data execução: {execution_date}")

In [0]:
import json
import requests
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, LongType, DoubleType
from azure.storage.blob import BlobServiceClient
from delta.tables import DeltaTable

storage_account = dbutils.secrets.get(scope='kv-scope-bitcoin', key='adls-storage-account-name')
storage_key     = dbutils.secrets.get(scope='kv-scope-bitcoin', key='adls-storage-key-bitcoin')

BRONZE_PATH = f'abfss://bronze@{storage_account}.dfs.core.windows.net/bitcoin/'

print('Configuração concluída.')

In [0]:
try:

    # Verifica se já existe dados na Bronze
    try:
        df_existente = spark.read.format('delta') \
            .option(f'fs.azure.account.key.{storage_account}.dfs.core.windows.net', storage_key) \
            .load(BRONZE_PATH)

        ultima_data_ms = df_existente.agg(F.max('timestamp_ms')).collect()[0][0]
        is_first_load = False
        print(f'Carga incremental. Último timestamp: {ultima_data_ms}')

    except:
        is_first_load = True
        ultima_data_ms = None
        print('Primeira carga. Baixando histórico completo.')

    # Lê o JSON do container landing
    blob_service_client = BlobServiceClient(
        account_url=f'https://{storage_account}.blob.core.windows.net',
        credential=storage_key
    )
    blob_client = blob_service_client.get_blob_client(
        container='landing',
        blob='bitcoin_daily.json'
    )
    content = blob_client.download_blob().readall()
    data    = json.loads(content)

    print(f'Registros recebidos do landing: {len(data["prices"])}')

    # Monta lista de registros
    records = []
    for i in range(len(data['prices'])):
        ts = int(data['prices'][i][0])

        # Se incremental, filtra apenas registros mais novos
        if not is_first_load and ts <= ultima_data_ms:
            continue

        records.append({
            'timestamp_ms': ts,
            'preco':        float(data['prices'][i][1]),
            'market_cap':   float(data['market_caps'][i][1]),
            'volume':       float(data['total_volumes'][i][1])
        })

    print(f'Registros novos para processar: {len(records)}')

    if len(records) == 0:
        dbutils.notebook.exit('SUCCESS: Bronze sem novos registros. Dados já atualizados.')

    # Cria DataFrame Spark 
    schema = StructType([
        StructField('timestamp_ms', LongType(),   True),
        StructField('preco',        DoubleType(), True),
        StructField('market_cap',   DoubleType(), True),
        StructField('volume',       DoubleType(), True),
    ])

    df_bronze = spark.createDataFrame(records, schema)

    # Adiciona metadados de auditoria 
    df_bronze = df_bronze \
        .withColumn('ativo',           F.lit('BTC')) \
        .withColumn('moeda',           F.lit('USD')) \
        .withColumn('data_carga_lake', F.current_date()) \
        .withColumn('fonte',           F.lit('coingecko'))

    total_novos = df_bronze.count()

    # Grava: overwrite na primeira carga, MERGE nas seguintes
    if is_first_load:
        df_bronze.write.format('delta') \
            .mode('overwrite') \
            .option('mergeSchema', 'true') \
            .option(f'fs.azure.account.key.{storage_account}.dfs.core.windows.net', storage_key) \
            .save(BRONZE_PATH)
        print('Primeira carga — overwrite concluído.')

    else:
        delta_table = DeltaTable.forPath(spark, BRONZE_PATH)

        delta_table.alias('target').merge(
            df_bronze.alias('source'),
            'target.timestamp_ms = source.timestamp_ms'
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()

        print('Carga incremental — MERGE concluído.')

    # Registra tabela no catálogo
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS bronze.tb_bitcoin
        USING DELTA
        LOCATION '{BRONZE_PATH}'
    """)

    total_final = spark.read.format('delta') \
        .option(f'fs.azure.account.key.{storage_account}.dfs.core.windows.net', storage_key) \
        .load(BRONZE_PATH).count()

    print(f'Total na Bronze após carga: {total_final}')
    dbutils.notebook.exit(f'SUCCESS: Bronze concluído. Novos: {total_novos}. Total: {total_final}')

except Exception as e:
    error_msg = str(e)
    if 'SUCCESS' in error_msg:
        dbutils.notebook.exit(error_msg)
    print(f'ERRO no Bronze: {error_msg}')
    dbutils.notebook.exit(f'ERROR: Bronze falhou. {error_msg}')